# Notebook 44 — Option 3: wide-prior round 1 + multi-round refinement round 2

## Purpose

This is one of three notebooks (44/45/46) built 2026-09-06 to test the remaining
options for salvaging System II (Wu2003 recycle CSTR) fault classification, after
the narrow lognormal training prior was found to cause catastrophic classification
failure (macro-F1≈0.07 — see `HANDOFF.md`, `docs/Manuscript/system_ii_sbi_solidification_plan.md`).
**You are running this notebook, not Claude** (per your instruction, to save tokens) —
read the markdown cells carefully, they explain what each step does and what to watch for.

**The idea being tested**: combine the two levers investigated on 2026-09-05/06 that
each showed a *partial* signal on their own:

1. **Widen the training prior's coverage** (`mixed_wide_prior`: `alpha, beta_r, eta_col,
   xi_reb` all `Uniform` over their prior box, `z_A0_eff` kept narrow lognormal). This is
   the generalization of `mixed_alpha_uniform_prior` (alpha-only), which already improved
   classification macro-F1 from 0.07 to 0.11. Smoke-tested viable this session
   (~46-60% accept rate). **This notebook trains round 1 on this wider prior** — the
   working hypothesis is that round 1 will be *less biased* than the narrow-lognormal or
   alpha-only-mixed posteriors, since all four flow parameters now have real prior mass
   near the Wu2003 fault catalogue's actual operating points.

2. **Multi-round (sequential) SNPE refinement**, confirmed this session to be
   *dramatically cheaper to simulate* (99.5% accept rate vs. 37-60% for uniform-prior
   widening, because round-2 draws concentrate near a real, physically-plausible
   operating point). **BUT**: a same-session pilot (`scripts/pilot_multiround_sc2.py`)
   found that round 2 does **not** fix a *biased* round-1 posterior — it only refines
   within whatever neighborhood round 1 already believes is plausible. Since round 1 in
   that pilot was the narrow-ish `mixed_alpha_uniform_prior` posterior (itself still
   biased for `alpha` away from the true fault region), round 2 just reinforced the bias.

**This notebook's hypothesis**: if round 1 is trained on the wider `mixed_wide_prior`
instead (point 1), it should be substantially *less* biased to begin with, so a round-2
refinement per scenario (point 2) might now actually deliver the precision gain it failed
to deliver on top of a biased round 1. **If round-2 classification here doesn't
meaningfully beat round-1's own classification, that would replicate and generalize
today's negative finding** — i.e. would mean per-scenario refinement isn't the missing
piece and the bottleneck really is round-1 prior coverage alone (in which case Option 3
collapses into "just finish the wide-prior fix," no multi-round needed).

## Two `sbi` 0.26.1 library bugs found and fixed this session — DO NOT remove the fixes below

Both bugs looked completely silent (no errors, no warnings, "Neural network successfully
converged" printed normally) but silently produced an unchanged/wrong posterior. Full
diagnosis in `HANDOFF.md`'s 2026-09-05 multi-round entry. In short:

1. **Round-2 training crashes** (`ValueError: NaN/Inf present in prior eval`) if the
   round-2 proposal is sampled with `reject_outside_prior=False` — a few draws land just
   outside the declared prior box, and SNPE-C's atomic loss needs `prior.log_prob(theta)`
   finite for every training theta. **Fix: always use `reject_outside_prior=True`** when
   sampling round-2 *training* draws (this is separate from evaluation-time sampling,
   which still uses `reject_outside_prior=False` for speed, matching the rest of this
   project's convention).
2. **The classic `SNPE_C` trainer does not reset its early-stopping bookkeeping
   (`_val_loss`, `_best_val_loss`, `_epochs_since_last_improvement`) between rounds.**
   A second round's `.train()` call will silently compare its own loss (a *different
   scale*, since `discard_prior_samples=True` changes the atomic loss's contrast-set
   size) against a stale round-1 loss value, judge every round-2 epoch as "no
   improvement", and revert to round-1's untouched weights — with a completely normal-
   looking "converged after N epochs" message and zero errors. Confirmed directly by
   diffing network state dicts (exactly 0.0 max weight difference when this bug bites).
   **Fix: reset all three attributes immediately before every round's `.train()` call
   beyond the first** (see the `round2_refine()` function below — the three reset lines
   are load-bearing, not decorative).


In [1]:
import pathlib, pickle, sys, time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from scipy import stats as scipy_stats

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn
from cstr_sbi.recycle.priors import mixed_wide_prior, PRIOR_LOW_5D, PRIOR_HIGH_5D
from cstr_sbi.recycle.physics import PARAM_NAMES
from cstr_sbi.recycle.scenarios import list_closed_loop, FAULT_UNITS
from cstr_sbi.recycle.simulator import nominal_warm_start, deterministic_window, noisy_replicates
from cstr_sbi.recycle.summaries import compute_summaries, N_SUMMARIES_SC2
from sbi_pipeline import _structure_config, _simulate_summary
from regenerate_sc2_wideprior_bank import generate_bank  # reuse the exact, already-checkpointed bank-gen logic

DATA = ROOT / "data"
RESULTS = ROOT / "results"
SBI_LOGS = ROOT / "sbi-logs"
for d in (DATA, RESULTS, SBI_LOGS):
    d.mkdir(exist_ok=True)

STRUCTURE = "S-C.2"
NOISE_PCT = 0.003
ARCH, HIDDEN, TRANSFORMS = "zuko_nsf", 60, 3
MAX_EPOCHS, STOP_AFTER_EPOCHS = 120, 12
SEED = 0
SUFFIX = "_wideprior_stage1bfix"
N_TRAIN = 1000
N_SBC_SCREEN = 120  # a screen only -- per finding_sbc_power_eta_col, N>=400 needed to trust a pass
THRESH, Z0_NOM = 0.85, 0.90
N_REPS, N_POST = 30, 200  # matches the established classification protocol (nb31 / run_sc2_*_classification.py)

# --- Config: which scenarios get the (expensive) round-2 refinement ---
# Round-2 per scenario costs roughly what the 2026-09-05 W2 pilot cost (~45 min for
# N_ROUND2_TARGET=200 draws at the accept rate seen there). Running all 14 could take
# many hours. Start with the smaller subset below (spans every fault class) and expand
# to `list_closed_loop()` (all 14) once you've confirmed the pipeline runs cleanly.
RUN_ALL_14 = False
PILOT_SCENARIO_NAMES = [
    "W1_healthy", "W2_cat_mild", "W5_jacket_mild", "W7_col_eff_mild",
    "W9_reb_fouling", "W10_feed_impurity", "W12_snowball_compound", "W16_full_multi",
]
N_ROUND2_TARGET = 200
N_ROUND2_CAP_ATTEMPTS = 400

prior_wide = mixed_wide_prior()
print(f"Config: RUN_ALL_14={RUN_ALL_14}, N_ROUND2_TARGET={N_ROUND2_TARGET}")


Config: RUN_ALL_14=False, N_ROUND2_TARGET=200


## Step 1 — round-1 training bank (`mixed_wide_prior`)

Reuses `scripts/regenerate_sc2_wideprior_bank.py`'s exact `generate_bank()` function
(checkpointed every 250 accepted rows, with a `.progress.log` update every 100 accepted rows,
resumable if interrupted — safe to stop and
restart this cell). This is the expensive step: at the smoke-tested ~46-60% accept rate,
n_train=1000 + one N=120 SBC screen (1120 draws total) could take several hours. Run
under `caffeinate` if executing this notebook via `jupyter nbconvert` from the terminal.
Tail `data/wu2003_sbi_train_sc2_true_n1000_wideprior_stage1bfix.progress.log` in another terminal
to watch accepted-row progress live during nbconvert execution.


In [2]:
t0 = time.time()

# if data for a complete or partial run already exists read it from disk, otherwise generate it anew

if (DATA / f"wu2003_sbi_train_sc2_true_n{N_TRAIN}{SUFFIX}.npz").exists() and (RESULTS / f"sc2_sbc_set_n{N_SBC_SCREEN}{SUFFIX}.npz").exists():
    theta_train, summary_train = load_bank(
        DATA / f"wu2003_sbi_train_sc2_true_n{N_TRAIN}{SUFFIX}.npz", label="train")
    theta_sbc, summary_sbc = load_bank(
        RESULTS / f"sc2_sbc_set_n{N_SBC_SCREEN}{SUFFIX}.npz", label="sbc_screen")
    print(f"Loaded existing data from disk. theta_train={theta_train.shape}, summary_train={summary_train.shape}")
    print(f"theta_sbc={theta_sbc.shape}, summary_sbc={summary_sbc.shape}")
    # if the data is not complete generate the missing parts
    if not (DATA / f"wu2003_sbi_train_sc2_true_n{N_TRAIN}{SUFFIX}.npz").exists():
        theta_train, summary_train = generate_bank(
            DATA / f"wu2003_sbi_train_sc2_true_n{N_TRAIN}{SUFFIX}.npz", N_TRAIN, seed=20260901, label="train")
    if not (RESULTS / f"sc2_sbc_set_n{N_SBC_SCREEN}{SUFFIX}.npz").exists():
        theta_sbc, summary_sbc = generate_bank(
            RESULTS / f"sc2_sbc_set_n{N_SBC_SCREEN}{SUFFIX}.npz", N_SBC_SCREEN, seed=20260902, label="sbc_screen")  
    print(f"After potential generation: theta_train={theta_train.shape}, summary_train={summary_train.shape}")
    print(f"theta_sbc={theta_sbc.shape}, summary_sbc={summary_sbc.shape}")
    print(f"Step 1 done in {time.time()-t0:.0f}s.")
else:
    print("Data not found on disk, generating anew.")
    theta_train, summary_train = generate_bank(
        DATA / f"wu2003_sbi_train_sc2_true_n{N_TRAIN}{SUFFIX}.npz", N_TRAIN, seed=20260901, label="train")
    theta_sbc, summary_sbc = generate_bank(
        RESULTS / f"sc2_sbc_set_n{N_SBC_SCREEN}{SUFFIX}.npz", N_SBC_SCREEN, seed=20260902, label="sbc_screen")
    print(f"Step 1 done in {time.time()-t0:.0f}s. theta_train={theta_train.shape}, summary_train={summary_train.shape}")
    print(f"theta_sbc={theta_sbc.shape}, summary_sbc={summary_sbc.shape}")


Data not found on disk, generating anew.


[2026-09-07 19:36:51] [train] resuming from partial: 250/1000 rows, 630 attempts


/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


[2026-09-07 20:21:24] [train] accepted=300/1000, attempts=734 (accept_rate=0.409), elapsed=2674s, ~8.91s/draw, ETA=104.0min


/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


... (32 outputs left out)

[2026-09-08 22:13:17] [sbc_screen] accepted=100/120, attempts=211 (accept_rate=0.474), elapsed=5456s, ~54.56s/draw, ETA=18.2min


/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


[2026-09-08 22:30:07] [sbc_screen] saved: /Users/simo/inso-po-RD/cstr-model-optimisation/research_project/results/sc2_sbc_set_n120_wideprior_stage1bfix.npz, theta=(120, 5), summaries=(120, 74), wall_s=6466, accept_rate=0.478


Step 1 done in 96801s. theta_train=(1000, 5), summary_train=(1000, 74)
theta_sbc=(120, 5), summary_sbc=(120, 74)


## Step 2 — train round-1 posterior (seed 0)

Same recipe as every other S-C.2 posterior this session (`zuko_nsf`, 60/3,
`z_score_x='none'` -- required for `z_A0_eff`, see `HANDOFF.md`'s `24a` entry).


In [3]:
# train_round1_fresh(): trains a BRAND NEW round-1 SNPE object/posterior from the
# (already-generated, cached) round-1 bank. Deliberately does NOT reuse a shared
# `inference` object across calls -- each call returns an independent (inference,
# posterior) pair.
#
# This matters for Step 4 below: SNPE-C's `discard_prior_samples=True` only discards
# ROUND-0 (the initial prior-drawn bank) data -- it does NOT discard earlier
# *intermediate* rounds. If the SAME `inference` object were reused across the
# per-scenario loop in Step 4, each scenario's round-2 refinement would silently
# accumulate on top of every PREVIOUS scenario's round-2 data too (round 2 = scenario
# A's data, round 3 = scenario B's data added on top of A's, etc.) -- contaminating
# what's supposed to be an independent, scenario-focused refinement. Retraining round 1
# fresh per scenario is cheap (seconds, since the bank is already on disk) and avoids
# this entirely.

def train_round1_fresh(seed=SEED):
    torch.manual_seed(seed)
    inference = SNPE(prior=prior_wide, density_estimator=posterior_nn(
        model=ARCH, hidden_features=HIDDEN, num_transforms=TRANSFORMS, z_score_x="none"))
    inference.append_simulations(torch.tensor(theta_train), torch.tensor(summary_train), proposal=prior_wide)
    t0 = time.time()
    est1 = inference.train(max_num_epochs=MAX_EPOCHS, validation_fraction=0.1,
                            stop_after_epochs=STOP_AFTER_EPOCHS, show_train_summary=False)
    posterior1 = inference.build_posterior(est1)
    return inference, posterior1, time.time() - t0


posterior1_path = SBI_LOGS / f"wu2003_posterior_sc2_true_n{N_TRAIN}_zscorenone_seed{SEED}{SUFFIX}.pkl"
if posterior1_path.exists():
    with open(posterior1_path, "rb") as f:
        payload1 = pickle.load(f)
    posterior1 = payload1["posterior"]
    print(f"Loaded existing round-1 posterior: {posterior1_path}")
else:
    _, posterior1, train_wall_s = train_round1_fresh(SEED)
    print(f"round-1 trained in {train_wall_s:.0f}s")
    with open(posterior1_path, "wb") as f:
        pickle.dump({"posterior": posterior1, "structure": "S-C.2", "n_train": N_TRAIN,
                     "torch_seed": SEED, "z_score_x": "none", "prior": "mixed_wide_prior"}, f)


/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/sbi/inference/trainers/npe/npe_base.py:196: UserWarning: Data has extreme outliers in dimension(s) [15, 49, 50, 54] (beyond 10.0x IQR from quartiles). This may cause precision loss during z-scoring, where distinct values become indistinguishable. Consider removing outliers from your data or setting `z_score_x='none'` (though this may affect training).
  warn_if_invalid_for_zscoring(x)


 Training neural network. Epochs trained: 6

 Training neural network. Epochs trained: 13

 Training neural network. Epochs trained: 20

 Training neural network. Epochs trained: 27

 Training neural network. Epochs trained: 34

... (10 outputs left out)

 Training neural network. Epochs trained: 111

 Training neural network. Epochs trained: 118

 Training neural network. Epochs trained: 121round-1 trained in 5s


/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/sbi/inference/potentials/posterior_based_potential.py:58: UserWarning: The passed prior has no support property, transform will be constructed from mean and std. If the passed prior is supposed to be bounded consider implementing the prior.support property.
  theta_transform = mcmc_transform(
/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/sbi/utils/sbiutils.py:879: UserWarning: The passed discrete prior has no mean or stddev attribute, estimating them from samples to build affine standardizing transform.
  transform = prior_mean_std_transform(prior, device)


## Step 2b (optional but recommended) — quick SBC screen on round 1

**This is a screen only (N=120), not a confirmation** — per `finding_sbc_power_eta_col`,
small-N SBC can give false passes (this specifically bit `eta_col` before at N=200). Do
not treat a pass here as certified calibration; it's just a sanity check that round 1
isn't obviously broken before spending hours on 14 rounds of round-2 refinement.


In [4]:
def run_sbc_screen(posterior, theta_set, summary_set, n_post, seed):
    torch.manual_seed(seed)
    ranks = {name: [] for name in PARAM_NAMES}
    for th, x_obs in zip(theta_set, summary_set):
        with torch.no_grad():
            samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),
                                     show_progress_bars=False, reject_outside_prior=False).detach().cpu().numpy()
        for k, name in enumerate(PARAM_NAMES):
            ranks[name].append(int(np.sum(samp[:, k] < th[k])))
    out = {}
    for name in PARAM_NAMES:
        r = np.asarray(ranks[name], dtype=float)
        ks = scipy_stats.ks_1samp(r / n_post, scipy_stats.uniform.cdf)
        out[name] = float(ks.pvalue)
    return out

sbc_r1 = run_sbc_screen(posterior1, theta_sbc, summary_sbc, n_post=N_SBC_SCREEN, seed=500001)
print("Round-1 SBC screen (N=120, NOT a confirmation):")
for name, p in sbc_r1.items():
    print(f"  {name:10s} ks_pvalue={p:.4g}  {'PASS (screen)' if p > 0.05 else 'fail'}")


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1560536180.py:6: UserWarning: 6.7% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1560536180.py:6: UserWarning: 5.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1560536180.py:6: UserWarning: 10.8% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1560536180.py:6: UserWarning: 5.8% of

/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1560536180.py:6: UserWarning: 33.3% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1560536180.py:6: UserWarning: 9.2% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1560536180.py:6: UserWarning: 22.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1560536180.py:6: UserWarning: 14.2% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1560536180.py:6: UserWarning: 20.8% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1560536180.py:6: UserWarning: 10.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1560536180.py:6: UserWarning: 11.7% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1560536180.py:6: UserWarning: 7.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1560536180.py:6: UserWarning: 12.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),


Round-1 SBC screen (N=120, NOT a confirmation):
  alpha      ks_pvalue=0.9988  PASS (screen)
  beta_r     ks_pvalue=0.7866  PASS (screen)
  eta_col    ks_pvalue=0.9798  PASS (screen)
  xi_reb     ks_pvalue=4.269e-17  fail
  z_A0_eff   ks_pvalue=0.111  PASS (screen)


## Step 3 — round-1 classification baseline (all 14 scenarios)

Same protocol as `scripts/run_sc2_mixedprior_classification.py` (N_REPS=30, N_POST=200,
posterior-mass threshold classifier). This is the "before" number to compare round 2
against.


In [5]:
def sample_fault_unit(theta_row, thresh=THRESH, z0_nom=Z0_NOM):
    alpha, beta_r, eta_col, xi_reb, z_A0 = theta_row
    reactor_bad = (alpha < thresh) or (beta_r < thresh)
    column_bad = (eta_col < thresh) or (xi_reb < thresh)
    feed_bad = z_A0 < thresh * z0_nom
    n_bad = int(reactor_bad) + int(column_bad) + int(feed_bad)
    if n_bad == 0:
        return "healthy"
    if n_bad >= 2:
        return "multi"
    if reactor_bad:
        return "reactor"
    if column_bad:
        return "column"
    return "feed"


def classify_fault_recycle(samples):
    labels = [sample_fault_unit(row) for row in samples]
    probs = {u: float(np.mean([l == u for l in labels])) for u in FAULT_UNITS}
    return max(probs, key=probs.__getitem__), probs


def compute_classification_metrics(preds, trues, classes=FAULT_UNITS):
    n = len(classes)
    cm = np.zeros((n, n), dtype=int)
    c2i = {c: i for i, c in enumerate(classes)}
    for true, pred in zip(trues, preds):
        cm[c2i[true], c2i[pred]] += 1
    f1_per_class = {}
    for i, cls in enumerate(classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_per_class[cls] = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    macro_f1 = float(np.mean(list(f1_per_class.values())))
    accuracy = float(np.diag(cm).sum() / cm.sum()) if cm.sum() > 0 else 0.0
    return {"confusion_matrix": cm, "class_names": classes, "per_class_f1": f1_per_class,
            "macro_f1": macro_f1, "accuracy": accuracy}


def build_x_o_and_reps(sc, y0_sb, seed=20260705):
    t_h, raw_obs = deterministic_window(sc, structure="S-B", y0=y0_sb)
    t_h_arr = np.asarray(t_h)
    rng = np.random.default_rng(seed)
    reps = noisy_replicates(raw_obs, n_replicates=N_REPS, rng=rng, noise_pct=NOISE_PCT)
    theta_np = np.asarray(sc.theta(), dtype=np.float32)
    summaries = [compute_summaries(reps[i], STRUCTURE, t_h_arr, theta=theta_np, rng=rng) for i in range(N_REPS)]
    return theta_np, sc.fault_unit(), summaries


y0_sb = nominal_warm_start("S-B")
all_scenarios = list_closed_loop()
scenario_lookup = {sc.name: sc for sc in all_scenarios}
scenarios_to_run = all_scenarios if RUN_ALL_14 else [scenario_lookup[n] for n in PILOT_SCENARIO_NAMES]
print(f"Running {len(scenarios_to_run)}/{len(all_scenarios)} scenarios: {[s.name for s in scenarios_to_run]}")

round1_rep_rows = []
scenario_x_cache = {}  # reused by Step 4 so we don't re-simulate the deterministic window twice
for sc in scenarios_to_run:
    theta_np, true_class, summaries = build_x_o_and_reps(sc, y0_sb)
    scenario_x_cache[sc.name] = (theta_np, true_class, summaries)
    for i, s in enumerate(summaries):
        with torch.no_grad():
            samp = posterior1.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
                                      show_progress_bars=False, reject_outside_prior=False).detach().cpu().numpy()
        pred, probs = classify_fault_recycle(samp)
        round1_rep_rows.append({"scenario": sc.name, "replicate": i, "true_class": true_class,
                                 "pred_class": pred, **probs})
    print(f"  round1 {sc.name:22s} true={true_class}")

round1_df = pd.DataFrame(round1_rep_rows)
round1_metrics = compute_classification_metrics(round1_df["pred_class"].tolist(), round1_df["true_class"].tolist())
print(f"\n=== ROUND-1 BASELINE ({len(scenarios_to_run)} scenarios) ===")
print(f"macro-F1={round1_metrics['macro_f1']:.3f}  accuracy={round1_metrics['accuracy']:.3f}")
print("per-class F1:", {k: round(v, 3) for k, v in round1_metrics['per_class_f1'].items()})
round1_df.to_csv(RESULTS / "44_round1_wideprior_classification_per_replicate.csv", index=False)


Running 8/14 scenarios: ['W1_healthy', 'W2_cat_mild', 'W5_jacket_mild', 'W7_col_eff_mild', 'W9_reb_fouling', 'W10_feed_impurity', 'W12_snowball_compound', 'W16_full_multi']


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1073657103.py:67: UserWarning: 8.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior1.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1073657103.py:67: UserWarning: 10.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior1.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1073657103.py:67: UserWarning: 6.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior1.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1073657103.py:67: UserWarning: 7.0% of samp

  round1 W1_healthy             true=healthy


  round1 W2_cat_mild            true=reactor


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1073657103.py:67: UserWarning: 5.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior1.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1073657103.py:67: UserWarning: 6.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior1.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1073657103.py:67: UserWarning: 6.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior1.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1073657103.py:67: UserWarning: 7.0% of sampl

/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1073657103.py:67: UserWarning: 5.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior1.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),


... (7 outputs left out)

  round1 W10_feed_impurity      true=feed


  round1 W12_snowball_compound  true=multi


  round1 W16_full_multi         true=multi

=== ROUND-1 BASELINE (8 scenarios) ===
macro-F1=0.246  accuracy=0.358
per-class F1: {'healthy': np.float64(0.217), 'reactor': np.float64(0.237), 'column': np.float64(0.183), 'feed': 0.0, 'multi': np.float64(0.594)}


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/1073657103.py:67: UserWarning: 5.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior1.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),


## Step 4 — round-2 multi-round refinement, per scenario

`round2_refine()` implements the exact fixed recipe from `scripts/pilot_multiround_sc2_continue.py`
(this session's working, bug-fixed version). **Do not remove the `reject_outside_prior=True`
in the sampling loop or the three-attribute reset before `.train()`** — see the markdown
at the top of this notebook for why each is load-bearing.


In [6]:
def round2_refine(inference, posterior1, x_o, seed_sim=0, n_target=N_ROUND2_TARGET,
                   cap_attempts=N_ROUND2_CAP_ATTEMPTS):
    proposal2 = posterior1.set_default_x(x_o)
    ctrl, y0 = _structure_config(STRUCTURE)
    rng = np.random.default_rng(20260906000 + seed_sim)
    theta_r2, summary_r2 = [], []
    attempts = 0
    t0 = time.time()
    torch.manual_seed(20260906000 + seed_sim)
    while len(theta_r2) < n_target and attempts < cap_attempts:
        attempts += 1
        with torch.no_grad():
            # IMPORTANT: reject_outside_prior=True here (bug #1 fix) -- False caused a
            # ValueError deep inside SNPE-C's atomic loss during this session's pilot.
            th = proposal2.sample((1,), show_progress_bars=False, reject_outside_prior=True).numpy()[0]
        s = _simulate_summary(th, ctrl, y0, STRUCTURE, NOISE_PCT, rng, scenario_specific_warm_start=True)
        if s is None or not np.isfinite(s).all():
            continue
        theta_r2.append(th)
        summary_r2.append(s)
    accept_rate = len(theta_r2) / attempts if attempts else 0.0
    wall_s = time.time() - t0
    if len(theta_r2) < 20:
        print(f"    *** only {len(theta_r2)} round-2 draws accepted after {attempts} attempts -- skipping this scenario ***")
        return None, accept_rate, wall_s

    theta_r2 = np.asarray(theta_r2, dtype=np.float32)
    summary_r2 = np.asarray(summary_r2, dtype=np.float32)
    inference.append_simulations(torch.tensor(theta_r2), torch.tensor(summary_r2), proposal=proposal2)
    # IMPORTANT: bug #2 fix -- reset ALL THREE attributes, not just _best_val_loss.
    # Without this, round-2 training silently reverts to round-1's untouched weights
    # (confirmed via a 0.0 max-weight-diff check in this session's pilot).
    inference._val_loss = float("inf")
    inference._best_val_loss = float("inf")
    inference._epochs_since_last_improvement = 0
    est2 = inference.train(max_num_epochs=300, validation_fraction=0.1, stop_after_epochs=30,
                            discard_prior_samples=True, show_train_summary=False)
    posterior2 = inference.build_posterior(est2)
    return posterior2, accept_rate, wall_s


In [7]:
round2_rep_rows = []
round2_diag_rows = []
for sc in scenarios_to_run:
    theta_np, true_class, summaries = scenario_x_cache[sc.name]
    x_o = torch.tensor(summaries[0], dtype=torch.float32)  # representative replicate, matches the pilot's convention
    t0 = time.time()
    # Fresh round-1 (inference, posterior1_fresh) PER SCENARIO -- see train_round1_fresh()'s
    # docstring above for why reusing one `inference` object across scenarios would be wrong.
    inference_fresh, posterior1_fresh, r1_wall_s = train_round1_fresh(seed=SEED)
    posterior2, accept_rate, sim_wall_s = round2_refine(inference_fresh, posterior1_fresh, x_o, seed_sim=sc.id)
    round2_diag_rows.append({"scenario": sc.name, "true_class": true_class, "accept_rate": accept_rate,
                              "round1_retrain_wall_s": r1_wall_s, "sim_wall_s": sim_wall_s,
                              "total_wall_s": time.time() - t0, "skipped": posterior2 is None})
    if posterior2 is None:
        continue
    for i, s in enumerate(summaries):
        with torch.no_grad():
            samp = posterior2.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
                                      show_progress_bars=False, reject_outside_prior=False).detach().cpu().numpy()
        pred, probs = classify_fault_recycle(samp)
        round2_rep_rows.append({"scenario": sc.name, "replicate": i, "true_class": true_class,
                                 "pred_class": pred, **probs})
    print(f"  round2 {sc.name:22s} true={true_class}  accept_rate={accept_rate:.3f}  "
          f"wall_s={time.time()-t0:.0f}", flush=True)

round2_df = pd.DataFrame(round2_rep_rows)
round2_diag_df = pd.DataFrame(round2_diag_rows)
round2_diag_df.to_csv(RESULTS / "44_round2_wideprior_multiround_diagnostics.csv", index=False)
if len(round2_df) > 0:
    round2_metrics = compute_classification_metrics(round2_df["pred_class"].tolist(), round2_df["true_class"].tolist())
    print(f"\n=== ROUND-2 REFINED ({round2_diag_df['skipped'].eq(False).sum()}/{len(scenarios_to_run)} scenarios) ===")
    print(f"macro-F1={round2_metrics['macro_f1']:.3f}  accuracy={round2_metrics['accuracy']:.3f}")
    print("per-class F1:", {k: round(v, 3) for k, v in round2_metrics['per_class_f1'].items()})
    round2_df.to_csv(RESULTS / "44_round2_wideprior_multiround_classification_per_replicate.csv", index=False)
else:
    print("\nNo scenarios produced a usable round-2 posterior -- see 44_round2_wideprior_multiround_diagnostics.csv")


/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/sbi/inference/trainers/npe/npe_base.py:196: UserWarning: Data has extreme outliers in dimension(s) [15, 49, 50, 54] (beyond 10.0x IQR from quartiles). This may cause precision loss during z-scoring, where distinct values become indistinguishable. Consider removing outliers from your data or setting `z_score_x='none'` (though this may affect training).
  warn_if_invalid_for_zscoring(x)


 Training neural network. Epochs trained: 4

 Training neural network. Epochs trained: 7

 Training neural network. Epochs trained: 11

 Training neural network. Epochs trained: 14

 Training neural network. Epochs trained: 18

... (573 outputs left out)

 Neural network successfully converged after 222 epochs.

/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/sbi/inference/potentials/posterior_based_potential.py:58: UserWarning: The passed prior has no support property, transform will be constructed from mean and std. If the passed prior is supposed to be bounded consider implementing the prior.support property.
  theta_transform = mcmc_transform(
/Users/simo/inso-po-RD/cstr-model-optimisation/research_project/.venv/lib/python3.10/site-packages/sbi/utils/sbiutils.py:879: UserWarning: The passed discrete prior has no mean or stddev attribute, estimating them from samples to build affine standardizing transform.
  transform = prior_mean_std_transform(prior, device)
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_61325/524338864.py:18: UserWarning: 99.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior2.sample((N_POST,), x=torch.tensor(s, dtype=torc

  round2 W16_full_multi         true=multi  accept_rate=0.590  wall_s=11825



=== ROUND-2 REFINED (8/8 scenarios) ===
macro-F1=0.168  accuracy=0.275
per-class F1: {'healthy': 0.0, 'reactor': np.float64(0.36), 'column': np.float64(0.065), 'feed': 0.0, 'multi': np.float64(0.415)}


## Step 5 — round-1 vs round-2 comparison

**What to look for**: if round-2's macro-F1 is meaningfully *higher* than round-1's for
the same scenario subset, the hypothesis holds (a less-biased round 1 + multi-round
refinement is a viable path). If they're close or round-2 is *worse* (as happened in the
2026-09-05 pilot on top of a more-biased round 1), that would mean per-scenario
refinement isn't adding value beyond fixing round-1's own prior coverage — i.e. Option 3
collapses into "just finish the wide-prior fix, skip the multi-round step."


In [8]:
print("=" * 70)
print(f"ROUND 1 (baseline, wide prior, {len(scenarios_to_run)} scenarios):")
print(f"  macro-F1={round1_metrics['macro_f1']:.3f}  accuracy={round1_metrics['accuracy']:.3f}")
if len(round2_df) > 0:
    print(f"ROUND 2 (per-scenario multi-round refinement):")
    print(f"  macro-F1={round2_metrics['macro_f1']:.3f}  accuracy={round2_metrics['accuracy']:.3f}")
    delta = round2_metrics['macro_f1'] - round1_metrics['macro_f1']
    print(f"\nDelta (round2 - round1) macro-F1: {delta:+.3f}")
print("=" * 70)
print("\nAlso compare against the historical numbers for context:")
print("  full-uniform-prior SBI (nb31, pre-bugfix):  macro-F1 = 0.694")
print("  narrow-lognormal SBI (S-C.2 baseline):       macro-F1 = 0.07-0.08")
print("  alpha-only mixed-prior SBI:                  macro-F1 = 0.11-0.112")


ROUND 1 (baseline, wide prior, 8 scenarios):
  macro-F1=0.246  accuracy=0.358
ROUND 2 (per-scenario multi-round refinement):
  macro-F1=0.168  accuracy=0.275

Delta (round2 - round1) macro-F1: -0.078

Also compare against the historical numbers for context:
  full-uniform-prior SBI (nb31, pre-bugfix):  macro-F1 = 0.694
  narrow-lognormal SBI (S-C.2 baseline):       macro-F1 = 0.07-0.08
  alpha-only mixed-prior SBI:                  macro-F1 = 0.11-0.112
